# 02 — Tuning chung cho DNN, LSTM, CNN

Cả ba model được định nghĩa trong `models.py`, dùng chung `notebook_workflow.py` và `configs/tuning.json`. Dữ liệu là Log + Z-score riêng từng client, fit chỉ trên tập fit.

10 trial/model: trial đầu là cấu hình neo, 4 random, 5 TPE. Mỗi trial chạy 3 Local + Centralized + FL; chọn bằng F1 trung bình 12 đánh giá validation (6 Local chéo + 3 Centralized + 3 FL). Local/Centralized tối đa 45 epoch, early stopping val_loss patience 5 và restore best weights; FL 15×3, lấy round cuối. Không dùng test khi tuning.

DNN/LSTM dùng architecture là số units ba layer. CNN dùng filters Conv1–filters Conv2–units Dense, có kernel_size riêng. CNN coi 39 cột là trục feature, không phải chuỗi thời gian. LSTM dùng GPU cuDNN tanh/sigmoid.

Trial đầu DNN: **8–4–2, dropout 0,5, batch 1024, client Adam LR 0,0005, server Adam LR 0,02**. Đây là cấu hình khởi đầu; các trial tiếp theo vẫn tìm trong search space. LSTM và CNN dùng cấu hình neo riêng trong configs/tuning.json.

FL tổng hợp đồng đều: UNSW = BoT = CIC = 1/3. Trọng số này cố định cho mọi trial và cả ba mô hình, chỉ dùng khi server gộp cập nhật sau local training; server tiếp tục cập nhật global model bằng Adam.

In [2]:
from pathlib import Path
import os, sys, json
if os.name == 'nt':
    raise RuntimeError('Mở start_notebooks.cmd để dùng kernel WSL /opt/flids-tf.')
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p/'notebook_workflow.py').is_file() and (p/'configs/tuning.json').is_file()), None)
if PROJECT is None:
    raise FileNotFoundError('Mở notebook trong FL_IDS.')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
CONFIG = PROJECT/'configs/tuning.json'
from notebook_workflow import settings, budget
C = settings(CONFIG)
RUN_ROOT = PROJECT/C['run_directory']
print('Project:', PROJECT)
print('Python:', sys.executable)
print('Run:', RUN_ROOT)


Project: /mnt/c/Users/admin/Downloads/FL_IDS
Python: /opt/flids-tf/bin/python
Run: /mnt/c/Users/admin/Downloads/FL_IDS/runs/main


In [3]:
from notebook_workflow import tune, tuning_table, tuning_details
display(budget(CONFIG))
MAX_NEW_EVALUATIONS = None  # đặt 1 hoặc 2 nếu muốn chia đợt

{'models': ['dnn', 'lstm', 'cnn'],
 'trials_per_model': 10,
 'trial_1': {'dnn': {'architecture': '8-4-2',
   'dropout': 0.5,
   'batch_size': 1024,
   'client_learning_rate': 0.0005,
   'server_learning_rate': 0.02},
  'lstm': {'architecture': '12-6-3',
   'dropout': 0.4,
   'batch_size': 2048,
   'client_learning_rate': 0.001,
   'server_learning_rate': 0.05},
  'cnn': {'architecture': '8-16-8',
   'kernel_size': 3,
   'dropout': 0.2,
   'batch_size': 2048,
   'client_learning_rate': 0.0005,
   'server_learning_rate': 0.01}},
 'startup': 'anchor + 4 random trials with default budget',
 'remaining': '5 TPE trials with default budget',
 'fits_per_trial': '3 Local + 1 Centralized + 1 FL',
 'rounds': 15,
 'local_epochs': 3,
 'client_weighting': 'uniform',
 'client_weights': {'unsw': 0.3333333333333333,
  'bot': 0.3333333333333333,
  'cic': 0.3333333333333333},
 'baseline_epochs': 45,
 'threshold': 0.5,
 'early_stopping': {'monitor': 'val_loss',
  'patience': 5,
  'min_delta': 0.0,
  'rest

In [3]:
best_dnn = tune('dnn', CONFIG, max_new_evaluations=MAX_NEW_EVALUATIONS)
display(best_dnn)

[I 2026-10-23 23:59:34,826] A new study created in RDB with name: dnn

DNN evaluation 1/10, Optuna trial 0, params={'architecture': '8-4-2', 'dropout': 0.5, 'batch_size': 1024, 'client_learning_rate': 0.0005, 'server_learning_rate': 0.02}
2026-10-23 23:59:35.619213: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-23 23:59:35.619379: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-23 23:59:35.619408: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-23 23:59:37.835158: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN facto

{'study_identity': 'a918907e0ec7b70ed015ce689de6904b02ed6f517bb383caba216fd40749af66',
 'model': 'dnn',
 'trial': 9,
 'params': {'architecture': '16-8-4',
  'dropout': 0.4,
  'batch_size': 4096,
  'client_learning_rate': 0.002,
  'server_learning_rate': 0.01},
 'rank': [0.7695237209251039, 0.8481094127111827, 0.7709171313391866],
 'completed_trials': 10,
 'required_trials': 10,
 'ready_for_final': True,
 'test_used_for_selection': False,
 'objective': 'mean of 12 three-client comparison F1 values; Local/Centralized at minimum val_loss epoch, FL at final round'}

In [4]:
best_lstm = tune('lstm', CONFIG, max_new_evaluations=MAX_NEW_EVALUATIONS)
display(best_lstm)

[I 2026-10-24 01:44:18,857] A new study created in RDB with name: lstm

LSTM evaluation 1/10, Optuna trial 0, params={'architecture': '12-6-3', 'dropout': 0.4, 'batch_size': 2048, 'client_learning_rate': 0.001, 'server_learning_rate': 0.05}
2026-10-24 01:44:19.537129: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-24 01:44:19.537186: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-24 01:44:19.537205: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-24 01:44:21.587933: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN fac

{'study_identity': 'a918907e0ec7b70ed015ce689de6904b02ed6f517bb383caba216fd40749af66',
 'model': 'lstm',
 'trial': 7,
 'params': {'architecture': '16-8-4',
  'dropout': 0.3,
  'batch_size': 1024,
  'client_learning_rate': 0.001,
  'server_learning_rate': 0.075},
 'rank': [0.7585189050395194, 0.7279253409906676, 0.7606622087878455],
 'completed_trials': 10,
 'required_trials': 10,
 'ready_for_final': True,
 'test_used_for_selection': False,
 'objective': 'mean of 12 three-client comparison F1 values; Local/Centralized at minimum val_loss epoch, FL at final round'}

In [4]:
best_cnn = tune('cnn', CONFIG, max_new_evaluations=MAX_NEW_EVALUATIONS)
display(best_cnn)

[I 2026-10-24 08:24:22,369] Using an existing study with name 'cnn' instead of creating a new one.

CNN evaluation 9/10, Optuna trial 8, params={'architecture': '16-32-16', 'dropout': 0.2, 'batch_size': 2048, 'client_learning_rate': 0.0005, 'server_learning_rate': 0.02, 'kernel_size': 3}
2026-10-24 08:24:23.275256: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-24 08:24:23.275326: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-24 08:24:23.276415: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-24 08:24:26.994724: E tensorflow/compiler/xla/stream_executor/cud

{'study_identity': 'a918907e0ec7b70ed015ce689de6904b02ed6f517bb383caba216fd40749af66',
 'model': 'cnn',
 'trial': 10,
 'params': {'architecture': '4-8-4',
  'dropout': 0.1,
  'batch_size': 1024,
  'client_learning_rate': 0.0003,
  'server_learning_rate': 0.01,
  'kernel_size': 3},
 'rank': [0.8224010022235628, 0.8744021427204898, 0.8444676932676125],
 'completed_trials': 10,
 'required_trials': 10,
 'ready_for_final': True,
 'test_used_for_selection': False,
 'objective': 'mean of 12 three-client comparison F1 values; Local/Centralized at minimum val_loss epoch, FL at final round'}

In [5]:
MODEL = 'dnn'  # dnn / lstm / cnn
display(tuning_table(MODEL, CONFIG))
display(tuning_details(MODEL, CONFIG))

,trial,score,min_fl_f1,architecture,dropout,batch_size,client_learning_rate,server_learning_rate
0,0,0.703087,0.891351,8-4-2,0.5,1024,0.0005,0.0200
1,1,0.678057,0.475691,12-6-3,0.3,2048,0.0020,0.0050
2,2,0.698508,0.730736,8-4-2,0.3,4096,0.0020,0.0200
3,3,0.712709,0.865889,8-4-2,0.4,2048,0.0005,0.0200
4,4,0.720779,0.820022,12-6-3,0.4,4096,0.0020,0.0200
5,5,0.691126,0.621606,12-6-3,0.4,4096,0.0010,0.0075
6,6,0.766259,0.837095,16-8-4,0.4,4096,0.0020,0.0150
7,7,0.752387,0.851074,16-8-4,0.5,1024,0.0010,0.0150
8,9,0.769524,0.848109,16-8-4,0.4,4096,0.0020,0.0100
9,10,0.744314,0.822680,16-8-4,0.3,2048,0.0005,0.0100


,trial,scenario,dataset,samples,accuracy,precision,recall,f1,far,auc,tn,fp,fn,tp
0,0,local_unsw,bot,20949,0.672156,0.626373,0.919308,0.745082,0.596850,0.655491,4044,5987,881,10037
1,0,local_unsw,cic,62981,0.659088,0.587232,0.970903,0.731830,0.627721,0.672097,12213,20593,878,29297
2,0,local_bot,unsw,12705,0.778906,0.990811,0.602867,0.749621,0.006806,0.870514,5691,39,2770,4205
3,0,local_bot,cic,62981,0.544974,0.608002,0.141508,0.229582,0.083918,0.723726,30053,2753,25905,4270
4,0,local_cic,unsw,12705,0.331287,0.326092,0.204444,0.251322,0.514311,0.304989,2783,2947,5549,1426
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,10,centralized,bot,20949,0.913647,0.998141,0.835867,0.909825,0.001695,0.993405,10014,17,1792,9126
116,10,centralized,cic,62981,0.975929,0.996328,0.953273,0.974325,0.003231,0.984932,32700,106,1410,28765
117,10,federated,unsw,12705,0.827942,0.947319,0.727025,0.822680,0.049215,0.875524,5448,282,1904,5071
118,10,federated,bot,20949,0.894983,0.972777,0.821487,0.890754,0.025022,0.911468,9780,251,1949,8969


Log epoch/round hiện trực tiếp và lưu trong train.log. Fit hoàn tất được dùng lại; fit dở chạy lại từ seed ban đầu. Chỉ chạy notebook 03 cho model đã hoàn tất đủ trial. Muốn đổi ngân sách hoặc search space, sửa `configs/tuning.json` trước khi bắt đầu; study đã tồn tại cần run_directory mới.